# ESCI-S — inspeção de dados

Objetivo: entender a estrutura real do `shuttie/esci-s` (metadata estendida do Amazon ESCI,
**incluindo reviews**) e decidir se o canal de review é utilizável.

**Contexto do artefato:**
- Scrape de ~1.66M ASINs (91.5% dos 1.814.925 do ESCI original), feito em **janeiro/2023**. Congelado desde.
- Licença Apache-2.0 no repo. Os dados são scrape do site da Amazon — zona cinzenta para redistribuição.
  Não republique o dump; referencie a fonte.
- Formato: ndjson (1 objeto JSON por linha), zstd no full / gzip no sample.

**Estratégia deste notebook:** descobrir o schema, não assumir. Dado raspado é heterogêneo —
livros vs produtos, templates de página diferentes, campos ausentes.

## 0. Setup e download

In [5]:
%pip uninstall -y pyarrow
%pip install --force-reinstall --no-cache-dir pyarrow

Found existing installation: pyarrow 25.0.0
Uninstalling pyarrow-25.0.0:
  Successfully uninstalled pyarrow-25.0.0
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 15.4 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [6]:
# !pip install pandas numpy zstandard pyarrow tqdm

import gzip, json, hashlib, re, os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

DATA = Path("data"); DATA.mkdir(exist_ok=True)

### Amostra (10 MB, 4.400 produtos) — comece por aqui

O arquivo no GitHub está em LFS, então `raw.githubusercontent.com` devolve **um ponteiro de texto**,
não o arquivo. Use o endpoint `media.githubusercontent.com`.

Checksums que eu li direto do ponteiro LFS do repo, para você validar a integridade:
- `size`: 10.569.620 bytes
- `sha256`: `c88659e2ed4a9c3ed16487ef27a37ae8dc95226723d6b0ed5dcbd422d587eb55`

In [7]:
SAMPLE = DATA / "esci_s_sample.json.gz"
SAMPLE_URL = "https://media.githubusercontent.com/media/shuttie/esci-s/master/sample.json.gz"
EXPECTED_SHA = "c88659e2ed4a9c3ed16487ef27a37ae8dc95226723d6b0ed5dcbd422d587eb55"
EXPECTED_SIZE = 10_569_620

if not SAMPLE.exists():
    !curl -L -o {SAMPLE} {SAMPLE_URL}

size = SAMPLE.stat().st_size
sha = hashlib.sha256(SAMPLE.read_bytes()).hexdigest()

print(f"size:   {size:,} (esperado {EXPECTED_SIZE:,}) {'OK' if size == EXPECTED_SIZE else 'DIVERGENTE'}")
print(f"sha256: {sha}")
print(f"        {'OK' if sha == EXPECTED_SHA else 'DIVERGENTE -> o artefato mudou, ou veio ponteiro LFS'}")

# Sanity: se veio ponteiro LFS, o arquivo tem ~130 bytes e comeca com 'version https://git-lfs'
if size < 10_000:
    print("\n!! Veio ponteiro LFS, nao o arquivo. Conteudo:")
    print(SAMPLE.read_text()[:300])

size:   10,569,620 (esperado 10,569,620) OK
sha256: c88659e2ed4a9c3ed16487ef27a37ae8dc95226723d6b0ed5dcbd422d587eb55
        OK


### Full dataset (3.4 GB zstd, 1.66M produtos) — só depois que o sample fizer sentido

Não baixe isso ainda. Rode o notebook inteiro no sample primeiro; a estrutura é a mesma
e você economiza 3.4 GB enquanto ainda está decidindo se o projeto existe.

```python
# S3:     https://esci-s.s3.amazonaws.com/esci.json.zst
# GDrive: https://drive.google.com/file/d/1HZLeUg61GWU3A6xyUEIw9AJtR-AwK5aw/view

# Leitura em streaming, sem descomprimir pra disco:
# import zstandard as zstd
# with open("data/esci.json.zst", "rb") as fh:
#     reader = zstd.ZstdDecompressor().stream_reader(fh)
#     for line in io.TextIOWrapper(reader, encoding="utf-8"):
#         rec = json.loads(line)
```

O repo tem 3 commits e último push em jan/2023. **Cheque se o S3 ainda responde**
(`curl -I https://esci-s.s3.amazonaws.com/esci.json.zst`) antes de planejar em cima disso.
Se tiver morrido, o GDrive é o plano B e você deveria espelhar o arquivo em algum lugar seu.

## 1. Carregar

In [ ]:
# S3:     https://esci-s.s3.amazonaws.com/esci.json.zst
# GDrive: https://drive.google.com/file/d/1HZLeUg61GWU3A6xyUEIw9AJtR-AwK5aw/view

# Leitura em streaming, sem descomprimir pra disco:
import zstandard as zstd
with open("data/esci.json.zst", "rb") as fh:
    reader = zstd.ZstdDecompressor().stream_reader(fh)
    for line in io.TextIOWrapper(reader, encoding="utf-8"):
        rec = json.loads(line)

In [8]:
def load_ndjson_gz(path, limit=None):
    recs = []
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for i, line in enumerate(fh):
            if limit and i >= limit:
                break
            line = line.strip()
            if line:
                recs.append(json.loads(line))
    return recs

recs = load_ndjson_gz(SAMPLE)
print(f"{len(recs):,} registros")
print(json.dumps(recs[0], ensure_ascii=False, indent=2)[:1500])

4,487 registros
{
  "type": "product",
  "locale": "jp",
  "asin": "B083TX3PNH",
  "title": "[コロンブス] 水のいらない汚れ落とし スニーカーケアスプレーフォームシャンプー メンズ",
  "stars": "3.7 out of 5 stars",
  "ratings": "220 ratings",
  "category": [
    "Clothing, Shoes & Jewelry",
    "Shoe, Jewelry & Watch Accessories",
    "Shoe Care & Accessories",
    "Shoe Treatments & Polishes",
    "Shoe Cleaners"
  ],
  "attrs": {},
  "bullets": [
    "Date First Available ‏ : ‎ February 17, 2020",
    "ASIN ‏ : ‎ B084VW2SVY",
    "Amazon Bestseller: #2,483 in Clothing, Shoes & Jewelry ( See Top 100 in Clothing, Shoes & Jewelry ) #5 in Shoe Cleaners",
    "#5 in Shoe Cleaners",
    "Customer Reviews: 3.7 out of 5 stars 220 ratings"
  ],
  "description": "From the Manufacturer",
  "info": {},
  "reviews": [
    {
      "stars": "4.0 out of 5 stars",
      "title": "Nice",
      "date": "Reviewed in Japan on December 19, 2022",
      "text": "Its nice."
    },
    {
      "stars": "4.0 out of 5 stars",
      "title": "たっぷり使ったら！

In [32]:
df_recs = pd.DataFrame(recs)
df_recs[df_recs['locale'] == 'us']

,type,locale,asin,title,stars,ratings,category,attrs,bullets,description,...,reviews,price,formats,template,subtitle,author,desc,attr,review,error
1,product,us,B07C1NJF6T,Rekayla Open Toe Tie Up Ankle Wrap Flat Sandals for Women,4.2 out of 5 stars,"5,656 ratings","[Clothing, Shoes & Jewelry, Women, Shoes, Sandals, Flats]",{},[],From the brand Previous page REKAYLA is a fresh new brand that wants to provide a top quality product to women all o...,...,"[{'stars': '4.0 out of 5 stars', 'title': 'Fine for a few hours', 'date': 'Reviewed in the United States 🇺🇸 on Octob...",,{},shoes,NaN,NaN,NaN,NaN,NaN,NaN
3,product,us,B06XXXLJ6V,"FYY Leather Case with Mirror for Samsung Galaxy S8 Plus, Leather Wallet Flip Folio Case with Mirror and Wrist Strap ...",4.3 out of 5 stars,"1,116 ratings","[Cell Phones & Accessories, Cases, Holsters & Sleeves, Flip Cases]","{'Material': 'Faux Leather', 'Compatible Phone Models': 'Samsung Galaxy S8 Plus', 'Form Factor': 'Flip', 'Brand': 'F...","[RFID Technique: Radio Frequency Identification technology, through radio signals to identify specific targets and t...",Product Description Premium PU Leather Top quality. Made with Premium PU Leather. Receiver design. Accurate cut-out ...,...,"[{'stars': '3.0 out of 5 stars', 'title': 'very cute but....', 'date': 'Reviewed in the United States 🇺🇸 on Septembe...",,{},wireless,NaN,NaN,NaN,NaN,NaN,NaN
4,product,us,B07WP4RXHY,"YUEPIN U-Tube Clamp 304 Stainless Steel Hose Pipe Cable Strap Clips With Rubber Cushioned (1-21/32""(42mm)-10pcs)",4.7 out of 5 stars,54 ratings,"[Tools & Home Improvement, Power & Hand Tools, Hand Tools, Workholding Devices, Clamps, Pipe Clamps]","{'Material': '304 stainless steel with rubber', 'Brand': 'YUEPIN', 'Color': 'Silver', 'Style': 'Classic'}","[Make sure this fits by entering your model number., This clamp Suit for 1-21/32""(42mm) diameter pipe,Mounting hole ...","Product Description Specification: Material: 304 Stainless Steel,100% New Rubber Color: Silver Shape: U Shape Quanti...",...,"[{'stars': '5.0 out of 5 stars', 'title': 'Worked well', 'date': 'Reviewed in the United States 🇺🇸 on June 28, 2022'...",$9.99,{},home_improvement,NaN,NaN,NaN,NaN,NaN,NaN
5,product,us,B07VRZTK2N,"Apron for Women, Waterproof Adjustable Bib Cooking Aprons with Pocket-2 Side Coral Velvet Towels Stitched Durable Pi...",4.0 out of 5 stars,152 ratings,[],{},"[Make sure this fits by entering your model number., Velvet]",,...,"[{'stars': '4.0 out of 5 stars', 'title': 'Cheap, water resistant and convenient', 'date': 'Reviewed in the United S...",$11.99,{},home,NaN,NaN,NaN,NaN,NaN,NaN
6,book,us,1368026222,Bruce's Big Storm (Mother Bruce Series),4.9 out of 5 stars,963 ratings,"[Books, Children's Books, Science, Nature & How It Works]",NaN,NaN,NaN,...,"[{'stars': '5.0 out of 5 stars', 'title': 'Cute book', 'date': 'Reviewed in the United States on November 23, 2022',...",,"{'Kindle': '$10.99', 'Hardcover': '$12.06'}",book,"Hardcover – Picture Book, September 3, 2019",Ryan Higgins,"Bruce's home is already a full house. But when a big storm brings all his woodland neighbors knocking, he'll have to...","{'Grade level': 'Preschool - Kindergarten', 'Dimensions': '9.45 x 0.55 x 12.45 inches', 'Reading age': '2 - 6 years,...","Editorial Reviews Review PRAISE FOR WE DON'T EAT OUR CLASSMATES ""Higgins . . . knows how to make big, scary animals ...",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4479,product,us,B07QG9P71V,"ORDTBY iPhone 7/8 Waterproof Case, Underwater Full Sealed Cover IP68 Certified for Waterproof Snowproof Shockproof a...",4.2 out of 5 stars,"3,152 ratings","[Cell Phones & Accessories, Cases, Holsters & Sleeves, Basic Cases]","{'Material': 'Plastic', 'Compatible Phone Models': 'IPhone 7', 'Form Factor': 'Basic Case', 'Brand': 'ORDTBY', 'Colo...","[Compatible with iPhone 7/8., Waterproof case with tight full sealed body, protect your iPhone 7/8 in all-weather. 6...","Product Description It’s a fashionable p

## 2. Descoberta de schema

Nunca confie no primeiro registro. Conte a presença de cada chave sobre **todos** os registros,
e separe "chave ausente" de "chave presente mas vazia" — em dado raspado essas duas coisas
significam coisas diferentes (não existe no site vs. o parser não achou).

In [9]:
def is_empty(v):
    return v is None or v == "" or v == [] or v == {}

key_present = Counter()
key_nonempty = Counter()
key_types = defaultdict(Counter)

for r in recs:
    for k, v in r.items():
        key_present[k] += 1
        key_types[k][type(v).__name__] += 1
        if not is_empty(v):
            key_nonempty[k] += 1

n = len(recs)
schema = pd.DataFrame([
    {
        "campo": k,
        "presente_%": round(100 * key_present[k] / n, 1),
        "nao_vazio_%": round(100 * key_nonempty[k] / n, 1),
        "tipos": dict(key_types[k]),
    }
    for k in sorted(key_present, key=lambda x: -key_nonempty[x])
])
schema

,campo,presente_%,nao_vazio_%,tipos
0,type,100.0,100.0,{'str': 4487}
1,locale,100.0,100.0,{'str': 4487}
2,asin,100.0,100.0,{'str': 4487}
3,template,100.0,96.3,{'str': 4487}
4,title,96.3,96.3,{'str': 4321}
5,ratings,96.3,91.8,{'str': 4321}
6,category,96.3,91.8,{'list': 4321}
7,stars,96.3,91.6,{'str': 4321}
8,reviews,96.3,88.4,{'list': 4321}
9,bullets,89.5,81.1,{'list': 4015}


Leia essa tabela com atenção ao gap entre `presente_%` e `nao_vazio_%`.
Um campo 100% presente e 40% não-vazio é um campo que o scraper sempre emite e frequentemente
não conseguiu preencher. Esse é o número que importa para decidir se um canal é viável.

In [10]:
# Heterogeneidade: type e template particionam o dataset em populacoes diferentes
print("type:")
print(pd.Series([r.get("type") for r in recs]).value_counts(dropna=False), "\n")

print("template (top 15):")
print(pd.Series([r.get("template") for r in recs]).value_counts(dropna=False).head(15), "\n")

print("locale:")
print(pd.Series([r.get("locale") for r in recs]).value_counts(dropna=False))

type:
product    4015
book        306
error       166
Name: count, dtype: int64 

template (top 15):
apparel              669
kitchen              361
home                 247
sports               230
home_improvement     222
health_and_beauty    216
beauty               214
book                 213
shoes                189
toy                  181
                     166
ce                   151
automotive           142
office_product       134
grocery              107
Name: count, dtype: int64 

locale:
us    3010
jp     838
es     639
Name: count, dtype: int64


In [11]:
# O schema e o mesmo entre product e book? (o README diz que livros tem campos extras)
by_type = defaultdict(Counter)
for r in recs:
    t = r.get("type", "?")
    for k, v in r.items():
        if not is_empty(v):
            by_type[t][k] += 1

type_counts = Counter(r.get("type", "?") for r in recs)
cmp = pd.DataFrame({
    t: {k: round(100 * by_type[t][k] / type_counts[t], 1) for k in sorted(key_present)}
    for t in type_counts
}).fillna(0)
cmp.columns = [f"{c} (n={type_counts[c]}) %" for c in cmp.columns]
cmp

,product (n=4015) %,book (n=306) %,error (n=166) %
asin,100.0,100.0,100.0
attr,0.0,98.7,0.0
attrs,59.9,0.0,0.0
author,0.0,94.1,0.0
bullets,90.6,0.0,0.0
category,95.3,94.4,0.0
desc,0.0,92.5,0.0
description,87.6,0.0,0.0
error,0.0,0.0,100.0
formats,1.9,99.0,0.0


## 3. O canal de review

Esta é a seção que decide o projeto. Três perguntas, nessa ordem:

1. **Cobertura** — quantos produtos têm ≥1 review? (O 91.5% publicado é cobertura de *scrape*, não de review.)
2. **Volume** — quantas reviews por produto? A página exibe ~8-10, então o teto é baixo.
3. **Qualidade** — o texto é substantivo ou é "amei, recomendo"?

In [12]:
review_counts = [len(r.get("reviews") or []) for r in recs]
rc = pd.Series(review_counts)

print(f"produtos com >=1 review: {(rc > 0).sum():,} / {len(rc):,}  ({100*(rc>0).mean():.1f}%)")
print(f"total de reviews:        {rc.sum():,}")
print()
print("distribuicao de reviews por produto:")
print(rc.describe(percentiles=[.1, .25, .5, .75, .9, .95, .99]))
print()
print("contagem exata (top 15):")
print(rc.value_counts().sort_index().head(15))

produtos com >=1 review: 3,968 / 4,487  (88.4%)
total de reviews:        34,997

distribuicao de reviews por produto:
count    4487.000000
mean        7.799643
std         4.273916
min         0.000000
10%         0.000000
25%         5.000000
50%         8.000000
75%        12.000000
90%        13.000000
95%        13.000000
99%        13.000000
max        13.000000
dtype: float64

contagem exata (top 15):
0      519
1      167
2      116
3      114
4       95
5      118
6       98
7      120
8     1430
9      249
10     152
11     131
12      98
13    1080
Name: count, dtype: int64


In [13]:
# Estrutura interna de uma review: as chaves sao consistentes?
rev_keys = Counter()
n_revs = 0
for r in recs:
    for rev in (r.get("reviews") or []):
        n_revs += 1
        for k in rev:
            rev_keys[k] += 1

print(f"{n_revs:,} reviews no total\n")
for k, c in rev_keys.most_common():
    print(f"  {k:10s} {100*c/n_revs:5.1f}%")

34,997 reviews no total

  stars      100.0%
  title      100.0%
  date       100.0%
  text       100.0%


In [14]:
# Volume de texto: e substantivo o suficiente pra extracao de aspecto?
lens = pd.Series([
    len((rev.get("text") or ""))
    for r in recs for rev in (r.get("reviews") or [])
])
print("comprimento do texto da review (chars):")
print(lens.describe(percentiles=[.1, .25, .5, .75, .9, .99]))
print(f"\nreviews com texto vazio: {(lens == 0).sum():,} ({100*(lens==0).mean():.1f}%)")
print(f"reviews com < 50 chars:  {(lens < 50).sum():,} ({100*(lens<50).mean():.1f}%)")

# Amostra do que e curto vs. longo
short = [rev for r in recs for rev in (r.get("reviews") or []) if 0 < len(rev.get("text") or "") < 40]
print("\nExemplos de review curta:")
for rev in short[:5]:
    print("  -", repr(rev.get("text")))

comprimento do texto da review (chars):
count    34997.000000
mean       306.061520
std        546.096641
min          0.000000
10%         26.000000
25%         62.000000
50%        156.000000
75%        344.000000
90%        673.000000
99%       2416.400000
max      13576.000000
dtype: float64

reviews com texto vazio: 98 (0.3%)
reviews com < 50 chars:  7,054 (20.2%)

Exemplos de review curta:
  - 'Its nice.'
  - 'コスパ良いが思った程は綺麗にならない。'
  - '実は半信半疑だったが使ってみてびっくり!汚れがきれいに落ちてスッキリしました。'
  - '全く汚れが落ちない 笑えるほど何も変わらない…'
  - 'Color is perfect'


### 3.1 Os campos sujos

In [15]:
# 'stars': "4.0 out of 5 stars" -> 4.0
STARS_RE = re.compile(r"([\d.]+)\s+out of\s+5")

def parse_stars(s):
    if not s:
        return None
    m = STARS_RE.search(str(s))
    return float(m.group(1)) if m else None

# 'ratings': "1,116 ratings" -> 1116
RATINGS_RE = re.compile(r"([\d,\.]+)\s+ratings?")

def parse_ratings(s):
    if not s:
        return None
    m = RATINGS_RE.search(str(s))
    return int(m.group(1).replace(",", "").replace(".", "")) if m else None

# 'date': "Reviewed in the United States 🇺🇸n September 22, 2022"
#          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^   ^ o '\n' foi comido pelo scraper, sobrou 'n'
DATE_RE = re.compile(
    r"Reviewed in (?P<country>.+?)\s*[\U0001F1E6-\U0001F1FF]{0,2}\s*n?\s*"
    r"(?:on\s+)?(?P<date>[A-Z][a-z]+ \d{1,2}, \d{4})"
)

def parse_review_date(s):
    if not s:
        return None, None
    m = DATE_RE.search(str(s))
    if not m:
        return None, None
    country = m.group("country").strip()
    try:
        dt = pd.to_datetime(m.group("date"), format="%B %d, %Y")
    except Exception:
        dt = None
    return country, dt

# Teste contra o exemplo do README
tests = [
    "Reviewed in the United States \U0001F1FA\U0001F1F8n September 22, 2022",
    "Reviewed in Australia \U0001F1E6\U0001F1FAn May 21, 2022",
]
for t in tests:
    print(repr(t), "->", parse_review_date(t))

'Reviewed in the United States 🇺🇸n September 22, 2022' -> ('the United States', Timestamp('2022-09-22 00:00:00'))
'Reviewed in Australia 🇦🇺n May 21, 2022' -> ('Australia', Timestamp('2022-05-21 00:00:00'))


In [16]:
# Taxa de falha dos parsers no dado real -- se for alta, o regex esta errado, nao o dado
star_vals, date_vals, countries = [], [], []
for r in recs:
    for rev in (r.get("reviews") or []):
        star_vals.append(parse_stars(rev.get("stars")))
        c, d = parse_review_date(rev.get("date"))
        countries.append(c)
        date_vals.append(d)

sv = pd.Series(star_vals)
dv = pd.Series(date_vals)
print(f"stars parseadas: {sv.notna().mean()*100:.1f}%   distribuicao:")
print(sv.value_counts().sort_index())
print(f"\ndatas parseadas: {dv.notna().mean()*100:.1f}%")
if dv.notna().any():
    print(f"  range: {dv.min()} .. {dv.max()}")

# Se a taxa de falha for alta, olhe o que falhou:
failed = [rev.get("date") for r in recs for rev in (r.get("reviews") or [])
          if parse_review_date(rev.get("date"))[1] is None and rev.get("date")]
print(f"\n{len(failed):,} datas nao parseadas. Amostra:")
for f in failed[:8]:
    print("  ", repr(f))

stars parseadas: 85.8%   distribuicao:
1.0     1362
2.0      954
3.0     2685
4.0     7306
5.0    17723
Name: count, dtype: int64

datas parseadas: 85.8%
  range: 1999-12-28 00:00:00 .. 2022-12-30 00:00:00

4,967 datas nao parseadas. Amostra:
   'Revisado en España 🇪🇸 el 17 de febrero de 2018'
   'Revisado en España 🇪🇸 el 18 de septiembre de 2018'
   'Revisado en España 🇪🇸 el 21 de septiembre de 2019'
   'Revisado en España 🇪🇸 el 25 de agosto de 2019'
   'Revisado en España 🇪🇸 el 3 de enero de 2019'
   'Revisado en España 🇪🇸 el 16 de junio de 2019'
   'Revisado en España 🇪🇸 el 29 de noviembre de 2017'
   'Revisado en España 🇪🇸 el 14 de septiembre de 2017'


### 3.2 Contaminação de locale

O bloco de reviews da página da Amazon mistura reviews internacionais. No exemplo do próprio README,
um produto `locale: "us"` tem uma review "Reviewed in Australia".

Isso importa para você: se o canal de review for embeddado com um modelo em inglês e uma fração
das reviews estiver em outro idioma, esse é ruído que você precisa quantificar (ou filtrar) — e
documentar, não descobrir depois que o resultado saiu estranho.

In [17]:
cs = pd.Series([c for c in countries if c])
print("paises de origem das reviews:")
print(cs.value_counts().head(20))
print(f"\nfracao NAO-US: {100 * (~cs.str.contains('United States', na=False)).mean():.1f}%")

paises de origem das reviews:
the United States     19917
Canada                 2630
the United Kingdom     2147
Japa                   2101
Japan                  1965
Mexico                  637
Australia               139
Germany                 139
India                   111
Italy                    96
Singapore                40
France                   40
Spain                    26
Spai                     20
Brazil                   19
Sweden                    2
the Netherlands           1
Name: count, dtype: int64

fracao NAO-US: 33.7%


In [18]:
# Cruzando com o locale do produto: quantos produtos 'us' tem review estrangeira?
rows = []
for r in recs:
    revs = r.get("reviews") or []
    if not revs:
        continue
    ctys = [parse_review_date(rev.get("date"))[0] for rev in revs]
    ctys = [c for c in ctys if c]
    if not ctys:
        continue
    n_foreign = sum(1 for c in ctys if "United States" not in c)
    rows.append({
        "asin": r.get("asin"),
        "locale": r.get("locale"),
        "n_reviews": len(revs),
        "n_com_pais": len(ctys),
        "n_estrangeiras": n_foreign,
        "frac_estrangeira": n_foreign / len(ctys),
    })

loc = pd.DataFrame(rows)
print(f"produtos analisados: {len(loc):,}")
print(f"produtos com >=1 review estrangeira: {(loc.n_estrangeiras > 0).sum():,} ({100*(loc.n_estrangeiras>0).mean():.1f}%)")
print()
print(loc.groupby("locale")["frac_estrangeira"].agg(["count", "mean", "max"]))

produtos analisados: 3,415
produtos com >=1 review estrangeira: 2,116 (62.0%)

        count      mean  max
locale                      
jp        633  0.987037  1.0
us       2782  0.185045  1.0


## 4. Join com o ESCI base

O `esci-s` só tem metadata. Os julgamentos de relevância (E/S/C/I) estão no repo oficial.
Chave do join: `asin` (no esci-s) ↔ `product_id` (no ESCI).

O ESCI oficial também usa LFS, então precisa de `git-lfs` instalado:

```bash
git lfs install
git clone https://github.com/amazon-science/esci-data.git
```

Os parquets ficam em `esci-data/shopping_queries_dataset/`.

In [20]:
%pip install -q datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:

from datasets import load_dataset
import pandas as pd

judgments = load_dataset("milistu/amazon-esci-data", "queries")
products  = load_dataset("milistu/amazon-esci-data", "products")

df_examples = pd.concat([judgments["train"].to_pandas(),
                         judgments["test"].to_pandas()], ignore_index=True)
df_products = pd.concat([products["train"].to_pandas(),
                         products["test"].to_pandas()], ignore_index=True)

print(df_examples.shape, df_examples.columns.tolist())
print(df_products.shape, df_products.columns.tolist())

README.md:   0%|          | 0.00/5.77k [00:00<?, ?B/s]

queries/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 47.9MB            

queries/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

queries/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.7MB            

queries/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

products/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  139MB            

products/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1371823 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/443101 [00:00<?, ? examples/s]

(2621288, 10) ['example_id', 'query', 'query_id', 'product_id', 'product_locale', 'esci_label', 'small_version', 'large_version', 'split', '__index_level_0__']
(1814924, 9) ['product_id', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'product_locale', 'split', '__index_level_0__']


In [34]:
df_products.query("product_locale == 'us'")

,product_id,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale,split,__index_level_0__
124055,B003O0MNGC,Delta BreezSignature VFB25ACH 80 CFM Exhaust Bath Fan with Humidity Sensor,None,Virtually silent at less than 0.3 sones\nPrecision engineered with DC brushless motor for extended reliability\nEasi...,DELTA ELECTRONICS (AMERICAS) LTD.,White,us,train,167168
124056,B00MARNO5Y,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed Fan/Light Bathroom Ventilation Fan with White Trim Ring,None,Super quiet 80CFM energy efficient fan virtually disappears into the ceiling leaving only a recessed light in view\n...,Aero Pure,White,us,train,167169
124057,B011RX6PNO,"Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroom Fan with LED Light and Humidity Sensor, White Finish",None,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joists Or Greater\nCountry of Origin: China\nBrand name: Aero Pure\nItem ...",Aero Pure,White Finish,us,train,167170
124058,B01MZIK0PI,"Delta Electronics (Americas) Ltd. RAD80 Delta BreezRadiance Series 80 CFM Fan with Heater, 10.5W, 1.5 Sones",None,"Quiet operation at 1.5 Sones\nPrecision engineered with DC brushless motor for extended reliability, this Fan will o...",DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,us,train,167171
124059,B01N5Y6002,"Delta Electronics (Americas) Ltd. GBR80HLED Delta BreezGreenBuilder Series 80 CFM Fan/Dimmable H, LED Light, Dual Sp...",None,Ultra energy-efficient LED module (11-watt equivalent to 60-watt incandescent light) included. Main light output-850...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",us,train,167172
...,...,...,...,...,...,...,...,...,...
1798600,B08XX3NL14,"SAYFINE 5 Pack Plain Atheletic Shirts for Men, Moisture Wicking Mens T Shirt Tee Shirts with Crew Neck, Short Sleeve...",<b>SAYFINE MENS T SHIRTS & GYM T SHIRTS & WORKOUT T SHIRTS <br><br> Men's Quick Dry Running T-Shirt:<br><br> Short S...,[VALUE PACK] - Athletic crew neck and flat-lock seam provide excellent comfortable and durability.High-quality and d...,SAYFINE,Black (5 Pack),us,test,1753347
1798601,B08Y8M9Z6L,GEEK LIGHTING Mens Polo Shirt Sport Casual Short Sleeve Golf Tennis T-Shirt,None,Material: This mens running shirt is made with 100% polyester fabric to keep you comfortable through any athletic ac...,GEEK LIGHTING,012-grey,us,test,1753348
1798602,B0936FKSS6,"Alex Vando Mens Golf Shirt Moisture Wicking Quick-Dry Short Sleeve Casual Polo Shirts for Men,Blue,XL",None,"Material: Our Mens Golf Shirts are made from 92%Polyester 8%Spandex, Light Weight, UPF40+, Quick Dry, Moisture Wicki...",Alex Vando,Blue,us,test,1753349
1798603,B096RXY1VT,"Casei Womens Polo Shirts Golf Shirts Quick Dry Moisture Wicking Black and White Polo Shirt （White,XL",black friday deals 2021 gifts for men gifts for women womens fall clothes halloween,【Quick Dry & Moisture Wicking】Womens polo shirts are technical fabric wicks moisture away from your skin and dries q...,Casei,White,us,test,1753350


In [35]:
df_examples.query("product_locale == 'us'")

,example_id,query,query_id,product_id,product_locale,esci_label,small_version,large_version,split,__index_level_0__
0,0,revent 80 cfm,0,B000MOO21W,us,I,0,1,train,0
1,1,revent 80 cfm,0,B07X3Y6B1V,us,E,0,1,train,1
2,2,revent 80 cfm,0,B07WDM7MQQ,us,E,0,1,train,2
3,3,revent 80 cfm,0,B07RH6Z8KW,us,E,0,1,train,3
4,4,revent 80 cfm,0,B07QJ7WYFQ,us,E,0,1,train,4
...,...,...,...,...,...,...,...,...,...,...
2619638,2614589,香奈儿,130378,B00NAGVL7W,us,E,1,1,test,2614589
2619639,2614590,香奈儿,130378,B00HJZT0YQ,us,I,1,1,test,2614590
2619640,2614591,香奈儿,130378,B006IB5T4W,us,I,1,1,test,2614591
2619641,2614592,香奈儿,130378,B0010POWEE,us,I,1,1,test,2614592


In [3]:
# Recorte: task 1 (ranking, small_version==1), locale us
task1 = df_examples[
    (df_examples.small_version == 1) & (df_examples.product_locale == "us")
].copy()

print(f"task1/us: {len(task1):,} julgamentos, "
      f"{task1.query_id.nunique():,} queries, "
      f"{task1.product_id.nunique():,} produtos")
print()
print("split:")
print(task1.split.value_counts())
print()
print("distribuicao de labels:")
print(task1.esci_label.value_counts(normalize=True).mul(100).round(1))

task1/us: 601,354 julgamentos, 29,844 queries, 482,105 produtos

split:
split
train    419653
test     181701
Name: count, dtype: int64

distribuicao de labels:
esci_label
E    43.5
S    35.1
I    16.9
C     4.5
Name: proportion, dtype: float64


In [19]:
# Cobertura do esci-s sobre o recorte -- no SAMPLE isso vai ser ~0.25%, e esperado.
# O numero que interessa e este mesmo codigo rodando no dump full.
s_asins = {r.get("asin") for r in recs if r.get("asin")}
s_with_reviews = {r.get("asin") for r in recs if (r.get("reviews") or [])}

t1_asins = set(task1.product_id.unique())

print(f"ASINs no task1/us:            {len(t1_asins):,}")
print(f"ASINs no esci-s (amostra):    {len(s_asins):,}")
print(f"  interseccao:                {len(t1_asins & s_asins):,}")
print(f"  interseccao COM review:     {len(t1_asins & s_with_reviews):,}")
print()
print("No dump full o esperado e ~91.5% de cobertura de scrape.")
print("A pergunta em aberto e a linha 'COM review' -- esse numero nao esta publicado.")

ASINs no task1/us:            482,105
ASINs no esci-s (amostra):    4,487
  interseccao:                1,241
  interseccao COM review:     1,146

No dump full o esperado e ~91.5% de cobertura de scrape.
A pergunta em aberto e a linha 'COM review' -- esse numero nao esta publicado.


## 5. O piso da avaliação

Antes de qualquer arquitetura: entenda por que o NDCG do ESCI tem piso alto.

O `Random` reportado no paper do SQID dá **NDCG 0.7483**. Não é bug — é consequência da
distribuição de labels. Cada query traz até 40 candidatos já pré-filtrados pela busca da Amazon,
e a maioria é `E` ou `S`. Ranquear aleatoriamente uma lista majoritariamente relevante já
pontua alto.

Consequência prática: sua faixa útil é ~0.75 → 0.86. Rode a célula abaixo para ver de onde
isso vem no seu recorte.

In [20]:
# Ganhos usados no benchmark oficial. ATENCAO: o paper do SQID (arXiv 2405.15190) reporta que o
# script oficial 'ranking/prepare_trec_eval_files.py' (linha 48) TROCA os ganhos de S e C.
# Confira a linha antes de usar; numeros publicados podem nao ser comparaveis entre si.
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}

def ndcg(gains, k=None):
    g = np.asarray(gains, dtype=float)
    if k: g = g[:k]
    disc = 1.0 / np.log2(np.arange(2, len(g) + 2))
    return float((g * disc).sum())

rng = np.random.default_rng(42)
rows = []
for qid, grp in task1.groupby("query_id"):
    gains = grp.esci_label.map(GAIN).values
    ideal = ndcg(sorted(gains, reverse=True))
    if ideal == 0:
        continue
    perm = rng.permutation(gains)
    rows.append({
        "query_id": qid,
        "n_cand": len(gains),
        "frac_E": (grp.esci_label == "E").mean(),
        "ndcg_random": ndcg(perm) / ideal,
    })

qs = pd.DataFrame(rows)
print(f"queries: {len(qs):,}")
print(f"\nNDCG de ranking ALEATORIO: {qs.ndcg_random.mean():.4f}")
print(f"candidatos por query: mediana {qs.n_cand.median():.0f}, media {qs.n_cand.mean():.1f}")
print(f"fracao de 'E' por query: mediana {qs.frac_E.median():.2f}")
print("\nSe esse numero bater ~0.75, voce reproduziu o piso do SQID e entendeu a escala do problema.")

queries: 29,844

NDCG de ranking ALEATORIO: 0.7444
candidatos por query: mediana 16, media 20.1
fracao de 'E' por query: mediana 0.42

Se esse numero bater ~0.75, voce reproduziu o piso do SQID e entendeu a escala do problema.


In [21]:
# Bootstrap: qual diferenca de NDCG e distinguivel de ruido com esse N de queries?
# Rode isso ANTES de desenhar o experimento -- define se o projeto consegue medir o que promete.
def bootstrap_ci(values, n_boot=2000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    vals = np.asarray(values)
    means = np.array([rng.choice(vals, size=len(vals), replace=True).mean()
                      for _ in range(n_boot)])
    return np.percentile(means, [100*alpha/2, 100*(1-alpha/2)])

lo, hi = bootstrap_ci(qs.ndcg_random.values)
print(f"NDCG random: {qs.ndcg_random.mean():.4f}  IC95% [{lo:.4f}, {hi:.4f}]")
print(f"largura do IC: {hi-lo:.4f}")
print()
print("Regra de bolso: se o ganho que voce reportar for menor que a largura desse IC,")
print("voce nao mediu nada. Bootstrap SOBRE QUERIES, nunca sobre pares query-produto -- ")
print("pares da mesma query nao sao independentes.")

NDCG random: 0.7444  IC95% [0.7425, 0.7465]
largura do IC: 0.0039

Regra de bolso: se o ganho que voce reportar for menor que a largura desse IC,
voce nao mediu nada. Bootstrap SOBRE QUERIES, nunca sobre pares query-produto -- 
pares da mesma query nao sao independentes.


## 6. Checklist go/no-go

Preencha com os números que saíram acima (do **dump full**, não do sample):

| # | Pergunta | Corte | Resultado |
|---|---|---|---|
| 1 | S3 do esci-s ainda responde? | 200 OK | |
| 2 | sha256 do sample confere? | exato | |
| 3 | % de ASINs do task1/us presentes no esci-s | >80% | |
| 4 | % de ASINs do task1/us **com ≥1 review** | >50% | |
| 5 | Mediana de reviews por produto | ≥5 | |
| 6 | Mediana de chars por review | ≥100 | |
| 7 | % de reviews não-US (contaminação de idioma) | <15% | |
| 8 | NDCG do ranking aleatório reproduz ~0.7483? | sim | |
| 9 | Largura do IC95% no bootstrap | <0.01 | |

**Se 4 ou 5 falharem, o canal de review não existe** e o projeto vira "multi-field sobre campos
de catálogo" — que ainda é um bom portfólio, mas é outra tese, e o CHARM (arXiv 2501.18707)
já é o prior work a citar.

**Se 9 falhar**, você ainda pode construir o sistema, mas reporte intervalos e não afirme
superioridade. Isso é mais honesto que 95.2% de Recall@3 com 4 produtos — e o teu diferencial
neste portfólio é exatamente esse, não o número.